In [1]:
from anndata import AnnData
from scipy.spatial import Delaunay
from scipy.stats import pearsonr, spearmanr, wilcoxon

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

## Load ST and scRNA datasets

In [ ]:
id_list = {
    'CID4290': 'ER_0',
    'CID4535': 'ER_1',
    'CID44971': 'TNBC_3',
    'CID4465': 'TNBC_2'
}
sample_list = id_list.keys()
adatas = dict()
adatas_sc = dict()
for sample in sample_list:
    adatas[sample] = sc.read_h5ad("results_breast/breast_st_stan/{}.h5ad".format(sample))
    
    adatas_sc[sample] = sc.read_h5ad("results_breast/breast_sc_ridge/{}.h5ad".format(sample))

In [ ]:
adatas_tfa = dict()
for sample in sample_list:
    adata = adatas[sample]
    adata_tfa = AnnData(
        X = adata.obsm['tfa_stan'],
        obs = adata.obs,
        obsm = {name: obj for (name, obj) in adata.obsm.items() if "tf" not in name},
        layers = {name: obj for (name, obj) in adata.obsm.items() if "tf" in name})
    adata_tfa.uns = adata.uns
    adatas_tfa[sample] = adata_tfa
    sc.pp.scale(adatas_tfa[sample])
    
adatas_sc_tfa = dict()
for sample in sample_list:
    adata = adatas_sc[sample]
    adata_tfa = AnnData(
        X = adata.obsm['tfa_ridge'],
        obs = adata.obs,
        obsm = {name: obj for (name, obj) in adata.obsm.items() if "tf" not in name},
        layers = {name: obj for (name, obj) in adata.obsm.items() if "tf" in name})
    adata_tfa.uns = adata.uns
    adatas_sc_tfa[sample] = adata_tfa
    sc.pp.scale(adatas_sc_tfa[sample])

In [ ]:
for sample in sample_list:
    df = pd.read_csv('results_breast/celltype_major/{}.csv'.format(sample), index_col=0)
    cells = np.intersect1d(adatas[sample].obs.index, df.index)
    
    df = df.loc[cells,:]
    adatas[sample] = adatas[sample][cells, :]
    adatas_tfa[sample] = adatas_tfa[sample][cells, :]
    
    adatas[sample].obsm['celltype'] = df
    adatas_tfa[sample].obsm['celltype'] = df

## Computing TF activity scores by cell type

In [ ]:
sc_scores = dict()
st_scores = dict()
st_scores_ridge = dict()

for sample in ["CID4290", "CID4465", "CID4535", "CID44971"]:
    print("-"*20, sample, "-"*20)

    adata_st = adatas_tfa[sample]
    adata_sc = adatas_sc_tfa[sample]

    var_names = np.intersect1d(adata_sc.var_names,adata_st.var_names)
    adata_sc = adata_sc[:,var_names]
    adata_st = adata_st[:,var_names]

    cts = np.intersect1d(adata_st.obsm['celltype'].columns, adata_sc.obs['celltype_major'].unique())

    adata_st.varm['ct_scores'] = pd.DataFrame(
        np.linalg.pinv(adata_st.obsm['celltype']).dot(adata_st.to_df('tfa_stan')).T,
        index=adata_st.var_names,
        columns=adata_st.obsm['celltype'].columns
    )
    adata_st.varm['ct_scores_ridge'] = pd.DataFrame(
        np.linalg.pinv(adata_st.obsm['celltype']).dot(adata_st.to_df('tfa_ridge')).T,
        index=adata_st.var_names,
        columns=adata_st.obsm['celltype'].columns
    )
    adata_sc.varm['ct_scores'] = adata_sc.to_df('tfa_ridge').groupby(adata_sc.obs['celltype_major']).mean().T

    print(pd.DataFrame({ "sc vs stl":
                       {ct: round(spearmanr(adata_sc.varm['ct_scores'][ct], adata_st.varm['ct_scores_ridge'][ct])[0],2)
                        for ct in cts},
                   "sc vs stan":{ct:  round(spearmanr(adata_sc.varm['ct_scores'][ct], adata_st.varm['ct_scores'][ct])[0],2)
                                 for ct in cts}}))


    for ct in cts:
        if sc_scores.get(ct) is None:
            sc_scores[ct]=dict()
        if st_scores.get(ct) is None:
            st_scores[ct]=dict()
        if st_scores_ridge.get(ct) is None:
            st_scores_ridge[ct]=dict()
        sc_scores[ct][sample] = adata_sc.varm['ct_scores'][ct]
        st_scores[ct][sample] = adata_st.varm['ct_scores'][ct]
        st_scores_ridge[ct][sample] = adata_st.varm['ct_scores_ridge'][ct]


In [ ]:
st_scores_list = dict()
sc_scores_list = dict()
st_scores_ridge_list = dict()
for sample in ["CID4290", "CID4465", "CID4535", "CID44971"]:
    df1 = pd.DataFrame([st_scores[x].get(sample) for x in st_scores.keys() if st_scores[x].get(sample) is not None]).T
    df2 = pd.DataFrame([sc_scores[x].get(sample) for x in sc_scores.keys() if sc_scores[x].get(sample) is not None]).T 
    df3 = pd.DataFrame([st_scores_ridge[x].get(sample) for x in st_scores_ridge.keys() if st_scores_ridge[x].get(sample) is not None]).T
    
    st_scores_list[sample] = (df1-df1.mean())/df1.std()
    sc_scores_list[sample] = (df2-df2.mean())/df2.std()
    st_scores_ridge_list[sample] = (df3-df3.mean())/df3.std()

In [ ]:
st_scores_list[sample]

In [ ]:
sc_scores_list[sample]

## Correlating samples

In [ ]:
df_corr = pd.DataFrame(index=id_list.values(),
                       columns=st_scores.keys())
for sample in sample_list:
    st_scores_df = st_scores_list[sample]
    sc_scores_df = sc_scores_list[sample]
    for celltype in sc_scores_df.columns:
        x = st_scores_df[celltype]
        y = sc_scores_df[celltype]
        df_corr.loc[id_list[sample], celltype] = spearmanr(x, y, alternative='two-sided')[0]

In [ ]:
df_corr = df_corr.fillna(0)
df_corr

In [ ]:
figsize = 5
fontsize = 18
plt.figure(figsize=(figsize*2.25,figsize*0.9), dpi=300)
plt.rc('font', size=fontsize)
ax = sns.heatmap(df_corr.round(2), cmap="RdBu_r", center=0, annot=True, square=True, vmax=1, vmin=-0.2)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0, ha='right')
plt.title(r'Spearman $\rho$', fontsize=fontsize, loc='right')